# 02b — Контроль alignment'а: shuffled lyrics + чужой текст

Гипотеза: `mean_confidence` в `forced_align` падает, когда лирика не соответствует аудио. Это ключевой сигнал Shazam-hybrid pipeline'а — без него мы не сможем отличить «нашли правильную лирику» от «склеили любую лирику с любым vocal'ом».

Тесты:
1. **Same lyrics** — sanity, должно дать ту же цифру что и 02_alignment (~0.82).
2. **Shuffled words** — тот же словарь, тот же объём, неверный порядок. Ожидаем падение confidence.
3. **Other song lyrics** — лирика другой песни (Кино — Группа крови). Ожидаем drop в 2-3×.

Запускать на **Colab T4** после 02_alignment.

In [ ]:
import torch
assert torch.cuda.is_available()
print(torch.cuda.get_device_name(0))

In [ ]:
!git clone -q https://github.com/RezPoint/unbake-test.git
%cd unbake-test
!pip install -q transformers torchaudio librosa jiwer pydantic requests
!python notebooks/download_dataset.py

In [ ]:
import random
from src.align import align

TRACK = 'data/raw/ru/Pharaoh - Дико, например.m4a'

PHARAOH_LYRICS = """Самый редкий вид, но самый худший браконьер
Спорим, она вместит себе в глотку револьвер?
Спорим, что ты больше не покинешь этот сквер?
Я играю с ее киской, детка, где твой кавалер?
Я врубаю Дафт Панк, она крутит мне блант
Покидаем клуб, в руке Совиньон-блан
Отлип от ее губ, залипаю в экран
Я кидаю деньги в воздух — ты поймаешь их сам
Эй, детка, ты модель или подделка?
Сейчас не важно, важно только, что не целка
А если кто-то из твоих захочет знать
Я найду еще причины, чтобы положить их спать
Она качает гривой, нельзя быть такой игривой
Выдыхаю в потолок испарения сативы
Кажусь хуже, чем я есть, заставляю тебя есть
Я держу при себе суку, что не может с меня слезть
Это все дико, например. Мы курим дико, например.
Мы виснем дико, например. Ты не сумел, но я дико отымел.
Это все дико, например. Мы курим дико, например.
Мы виснем дико, например. Ты не сумел, но я дико отымел.
Извини за прямоту, но даже в этом клубе все когда-нибудь умрут
Мои парни мне сказали, что мне нужно охладить
И я держу это на низком, ведь та сука троглодит
Повсюду лужи крови, помогу ей обойти, но ты беги-беги-беги
Марриотт с меня хуеет, трижды место преступления
Моя тишка говорит мне: Я ошибка поколения
Это все дико, например. Мы курим дико, например.
Мы виснем дико, например. Ты не сумел, но я дико отымел.
Это все дико, например. Мы курим дико, например.
Мы виснем дико, например. Ты не сумел, но я дико отымел."""

# Кино — Группа крови (общеизвестная лирика, ~ та же длина)
OTHER_LYRICS = """Тёплое место, но улицы ждут отпечатков наших ног.
Звёздная пыль на сапогах, мягкое кресло, клетчатый плед,
Не нажатый вовремя курок, солнечный день в ослепительных снах.
Группа крови на рукаве, мой порядковый номер на рукаве,
Пожелай мне удачи в бою, пожелай мне не остаться в этой траве,
Пожелай мне удачи, пожелай мне удачи.
И есть чем платить, но я не хочу победы любой ценой,
Я никому не хочу ставить ногу на грудь.
Я хотел бы остаться с тобой, просто остаться с тобой,
Но высокая в небе звезда зовёт меня в путь.
Группа крови на рукаве, мой порядковый номер на рукаве,
Пожелай мне удачи в бою, пожелай мне не остаться в этой траве,
Пожелай мне удачи, пожелай мне удачи.
Группа крови на рукаве, мой порядковый номер на рукаве,
Пожелай мне удачи в бою, пожелай мне не остаться в этой траве,
Пожелай мне удачи, пожелай мне удачи."""

In [ ]:
# Test 1 — same lyrics (sanity)
_, t_same = align(TRACK, PHARAOH_LYRICS, language='ru')
print('SAME    :', t_same)

In [ ]:
# Test 2 — shuffled (тот же словарь, неверный порядок)
random.seed(42)
words = PHARAOH_LYRICS.split()
random.shuffle(words)
SHUFFLED = ' '.join(words)
_, t_shuf = align(TRACK, SHUFFLED, language='ru')
print('SHUFFLED:', t_shuf)

In [ ]:
# Test 3 — другая песня
_, t_other = align(TRACK, OTHER_LYRICS, language='ru')
print('OTHER   :', t_other)

In [ ]:
# Сводная таблица
import json, dataclasses
rows = [('same', t_same), ('shuffled', t_shuf), ('other_song', t_other)]
summary = {
    name: {
        'mean_confidence': t.mean_confidence,
        'coverage': t.coverage,
        'n_aligned': t.n_aligned,
        'n_lyric_words': t.n_lyric_words,
    } for name, t in rows
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
with open('/content/align_control.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print('saved → /content/align_control.json')

## Что искать

- **same**: ~0.82 (как в 02_alignment).
- **shuffled**: ожидаем 0.4-0.6. CTC всё равно «жадно» приклеивает символы к фреймам, но без структуры падает.
- **other_song**: ожидаем 0.2-0.4. Слова физически отсутствуют в аудио — confidence обвалится.

Если `same` >> `other_song` (хотя бы 2× разрыв) → **confidence работает как cover-detector**.
Если разрыв < 1.5× → нужен дополнительный сигнал (например, ASR-vs-alignment word-match-rate).